# Azalyst — Notebook 1: Build Feature Cache

**What this does:**
- Reads raw Binance parquet files (444 coins, 3yr)
- Builds 56 features per symbol, one at a time
- Uploads each `.parquet` directly to **Google Drive** immediately after building
- Deletes local copy right after upload → disk never exceeds ~400MB
- Resume-safe: already-uploaded files are automatically skipped

**Then run Notebook 2** (`azalyst_2_train.ipynb`) which downloads the cache from Google Drive and trains.

---
### One-time setup (do this before first run)
1. Go to [console.cloud.google.com](https://console.cloud.google.com) → Create a project
2. Enable **Google Drive API** for the project
3. Create a **Service Account** → Keys → Add Key → JSON → download the `.json` file
4. In **Google Drive**, create a folder named `azalyst-feature-cache` → open it → copy the folder ID from the URL  
   *(URL looks like `https://drive.google.com/drive/folders/YOUR_FOLDER_ID_HERE`)*
5. Share the Google Drive folder with the **service account email** (found in the JSON, ends in `@...iam.gserviceaccount.com`) — give it **Editor** access
6. In Kaggle Notebook → **Add-ons → Secrets** → add secret named `GDRIVE_SERVICE_KEY`  
   Value: paste the **entire contents** of the downloaded JSON key file
7. Set `GDRIVE_FOLDER_ID` in Cell 1 below to your folder ID


In [ ]:
# ── Cell 0: Install deps ─────────────────────────────────────────────────────
import subprocess, sys
for pkg in ['xgboost', 'lightgbm', 'polars', 'sortedcontainers',
            'google-api-python-client', 'google-auth']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Deps installed.')


In [ ]:
# ── Cell 1: CONFIG ────────────────────────────────────────────────────────────
import os, sys, time, gc, shutil, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

# ── SET THIS ──────────────────────────────────────────────────────────────────
GDRIVE_FOLDER_ID  = 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE'  # from Drive folder URL
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR          = '/kaggle/input/binance-data-5min-300-coins-3years'
FEATURE_STORE_DIR = '/kaggle/working/feature_cache'
BATCH_SIZE        = 50

os.makedirs(FEATURE_STORE_DIR, exist_ok=True)

all_files = sorted(Path(DATA_DIR).rglob('*.parquet'))
if not all_files:
    all_files = sorted(Path(DATA_DIR).parent.rglob('*.parquet'))

total_batches = -(-len(all_files) // BATCH_SIZE)

print(f'DATA_DIR       : {DATA_DIR}')
print(f'Total symbols  : {len(all_files)}')
print(f'Batches        : {total_batches}  ({BATCH_SIZE} symbols each)')
print(f'Drive folder   : {GDRIVE_FOLDER_ID}')
print(f'Disk free      : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB')


In [ ]:
# ── Cell 2: Google Drive API helpers ─────────────────────────────────────────
import json as _json
from kaggle_secrets import UserSecretsClient
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# Load service account credentials from Kaggle Secret
_secret = UserSecretsClient().get_secret('GDRIVE_SERVICE_KEY')
_creds  = service_account.Credentials.from_service_account_info(
    _json.loads(_secret),
    scopes=['https://www.googleapis.com/auth/drive']
)
drive = build('drive', 'v3', credentials=_creds)
print(f'✅ [Google Drive] Authenticated')

def gdrive_list_files():
    """Return set of filenames already in the Drive folder."""
    results = []
    page_token = None
    while True:
        resp = drive.files().list(
            q=f"'{GDRIVE_FOLDER_ID}' in parents and trashed=false",
            fields='nextPageToken, files(name)',
            pageToken=page_token
        ).execute()
        results.extend(resp.get('files', []))
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
    return {f['name'] for f in results}

def gdrive_upload_file(local_path):
    """Upload a single file to the Drive folder."""
    name  = Path(local_path).name
    media = MediaFileUpload(str(local_path), mimetype='application/octet-stream', resumable=True)
    drive.files().create(
        body={'name': name, 'parents': [GDRIVE_FOLDER_ID]},
        media_body=media
    ).execute()

# Verify folder is accessible
existing = gdrive_list_files()
print(f'  Files already in Drive folder : {len(existing)}')
print('[Cell 2] Google Drive helpers ready.')


In [ ]:
# ── Cell 3: Config + Feature Engineering ────────────────────────────────────
BARS_PER_HOUR = 12
BARS_PER_DAY  = 288
HORIZON_BARS  = 48   # 4H @ 5-min bars
CHUNK_GC_EVERY = 25

FEATURE_COLS = [
    'ret_1bar','ret_1h','ret_4h','ret_1d','ret_2d','ret_3d','ret_1w',
    'vol_ratio','vol_ret_1h','vol_ret_1d','obv_change','vpt_change','vol_momentum',
    'rvol_1h','rvol_4h','rvol_1d','vol_ratio_1h_1d','atr_norm','parkinson_vol','garman_klass',
    'rsi_14','rsi_6','macd_hist','bb_pos','bb_width','stoch_k','stoch_d','cci_14','adx_14','dmi_diff',
    'vwap_dev','amihud','kyle_lambda','spread_proxy','body_ratio','candle_dir',
    'wick_top','wick_bot','price_accel','skew_1d','kurt_1d','max_ret_4h',
    'wq_alpha001','wq_alpha012','wq_alpha031','wq_alpha098',
    'cs_momentum','cs_reversal','vol_adjusted_mom','trend_consistency',
    'vol_regime','trend_strength','corr_btc_proxy','hurst_exp','fft_strength',
    'frac_diff_close',
]

def _rsi(s, n):
    d = s.diff(); g = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    ls = (-d).clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    return 100 - 100 / (1 + g / ls.replace(0, np.nan))

def _ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def frac_diff_ffd(series, d=0.4, threshold=1e-5):
    w = [1.0]; k = 1
    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < threshold: break
        w.append(w_k); k += 1
    w = np.array(w[::-1], dtype=np.float64)
    width = len(w)
    arr = series.values.astype(np.float64)
    out = np.full(len(arr), np.nan, dtype=np.float64)
    for i in range(width - 1, len(arr)):
        chunk = arr[i - width + 1 : i + 1]
        if np.isnan(chunk).any(): continue
        out[i] = np.dot(w, chunk)
    return pd.Series(out, index=series.index, dtype=np.float32)

def build_features(df):
    bph, bpd = BARS_PER_HOUR, BARS_PER_DAY
    o, h, l, c, v = df['open'], df['high'], df['low'], df['close'], df['volume']
    f = pd.DataFrame(index=df.index)
    # Returns
    f['ret_1bar'] = c.pct_change()
    for name, bars in [('ret_1h',bph),('ret_4h',bph*4),('ret_1d',bpd),
                       ('ret_2d',bpd*2),('ret_3d',bpd*3),('ret_1w',bpd*7)]:
        f[name] = c.pct_change(bars)
    # Volume
    avg_v = v.rolling(bpd, min_periods=1).mean()
    f['vol_ratio']    = v / avg_v.replace(0, np.nan)
    f['vol_ret_1h']   = v.pct_change(bph)
    f['vol_ret_1d']   = v.pct_change(bpd)
    f['obv_change']   = (v * np.sign(c.diff())).cumsum().pct_change(bph)
    pt = v * c.pct_change()
    f['vpt_change']   = pt.cumsum().pct_change(bph)
    f['vol_momentum'] = v.rolling(bph).mean() / v.rolling(bpd).mean().replace(0, np.nan)
    # Volatility
    f['rvol_1h']  = c.pct_change().rolling(bph).std()
    f['rvol_4h']  = c.pct_change().rolling(bph*4).std()
    f['rvol_1d']  = c.pct_change().rolling(bpd).std()
    f['vol_ratio_1h_1d'] = f['rvol_1h'] / f['rvol_1d'].replace(0, np.nan)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    f['atr_norm']    = tr.rolling(bph*4).mean() / c.replace(0, np.nan)
    hl = np.log(h / l.replace(0, np.nan))
    co = np.log(c / o.replace(0, np.nan))
    f['parkinson_vol'] = hl.rolling(bpd).apply(lambda x: np.sqrt((x**2).mean() / (4*np.log(2))), raw=True)
    f['garman_klass']  = (0.5 * hl**2 - (2*np.log(2)-1) * co**2).rolling(bpd).mean()
    # Technical
    f['rsi_14'] = _rsi(c, 14) / 100
    f['rsi_6']  = _rsi(c, 6) / 100
    e12, e26 = _ema(c,bph*12), _ema(c,bph*26)
    macd_line = e12 - e26; signal_line = _ema(macd_line, 9*bph)
    f['macd_hist'] = (macd_line - signal_line) / c.replace(0, np.nan)
    bb_mid = c.rolling(20*bph).mean(); bb_std = c.rolling(20*bph).std()
    f['bb_pos']   = (c - bb_mid) / (2 * bb_std.replace(0, np.nan))
    f['bb_width'] = (4 * bb_std) / bb_mid.replace(0, np.nan)
    lo_n, hi_n = l.rolling(14*bph).min(), h.rolling(14*bph).max()
    f['stoch_k'] = (c - lo_n) / (hi_n - lo_n).replace(0, np.nan)
    f['stoch_d'] = f['stoch_k'].rolling(3*bph).mean()
    typical = (h + l + c) / 3
    f['cci_14'] = (typical - typical.rolling(14*bph).mean()) / (0.015 * typical.rolling(14*bph).std().replace(0, np.nan))
    pdi = ((h.diff().clip(lower=0)).ewm(span=14*bph).mean() / tr.ewm(span=14*bph).mean().replace(0, np.nan)) * 100
    ndi = ((-l.diff()).clip(lower=0).ewm(span=14*bph).mean() / tr.ewm(span=14*bph).mean().replace(0, np.nan)) * 100
    f['adx_14']  = (abs(pdi - ndi) / (pdi + ndi).replace(0, np.nan)).ewm(span=14*bph).mean()
    f['dmi_diff'] = (pdi - ndi) / 100
    # Microstructure
    vwap = (typical * v).cumsum() / v.cumsum().replace(0, np.nan)
    f['vwap_dev']    = (c - vwap) / vwap.replace(0, np.nan)
    abs_r = c.pct_change().abs()
    f['amihud']      = abs_r / (v * c).replace(0, np.nan)
    f['kyle_lambda'] = abs_r.rolling(bph).mean() / v.rolling(bph).mean().replace(0, np.nan)
    f['spread_proxy']= (h - l) / c.replace(0, np.nan)
    body = abs(c - o)
    f['body_ratio']  = body / (h - l).replace(0, np.nan)
    f['candle_dir']  = np.sign(c - o)
    # Price structure
    f['wick_top']    = (h - pd.concat([o,c],axis=1).max(axis=1)) / (h-l).replace(0, np.nan)
    f['wick_bot']    = (pd.concat([o,c],axis=1).min(axis=1) - l) / (h-l).replace(0, np.nan)
    f['price_accel'] = c.pct_change().diff()
    f['skew_1d']     = c.pct_change().rolling(bpd).skew()
    f['kurt_1d']     = c.pct_change().rolling(bpd).kurt()
    f['max_ret_4h']  = c.pct_change().rolling(bph*4).max()
    # WQ Alphas
    f['wq_alpha001'] = (-c.pct_change().rolling(bph).std().rank(pct=True)).fillna(0)
    f['wq_alpha012'] = (-np.sign(v.diff()) * c.pct_change()).fillna(0)
    delay_c = c.shift(bph)
    f['wq_alpha031'] = ((c-delay_c).rank(pct=True).rolling(bph*4).mean() * (-c.pct_change(bph*2)) * v.pct_change(bph)).fillna(0)
    f['wq_alpha098'] = (c.rolling(5*bph).mean().rank(pct=True) - c.pct_change(bph*2) * v.rank(pct=True)).fillna(0)
    # Cross-section + Regime
    f['cs_momentum']      = c.pct_change(bpd*5).rank(pct=True)
    f['cs_reversal']      = (-c.pct_change(bpd)).rank(pct=True)
    f['vol_adjusted_mom'] = f['ret_1w'] / f['rvol_1d'].replace(0, np.nan)
    r5 = c.pct_change(bpd*5)
    sign_same = r5.rolling(bpd*5, min_periods=bpd).apply(
        lambda x: (np.sign(x)==np.sign(x.iloc[-1])).mean(), raw=False)
    f['trend_consistency'] = sign_same
    f['vol_regime']        = f['rvol_1d'].rolling(bpd*5).rank(pct=True)
    f['trend_strength']    = abs(f['ret_1w']) / f['rvol_1d'].replace(0, np.nan)
    btc_proxy = c.rolling(bpd).corr(c.rolling(bpd).mean())
    f['corr_btc_proxy']    = btc_proxy
    # Hurst
    hurst_win = bpd * 5
    hurst_exp = pd.Series(np.nan, index=df.index, dtype=np.float32)
    close_arr = c.values.astype(np.float64)
    for i in range(hurst_win, len(close_arr)):
        seg = close_arr[i-hurst_win:i]; rs_vals = []
        for ss in [hurst_win//4, hurst_win//2, hurst_win]:
            sub = np.diff(np.log(seg[-ss:] + 1e-10))
            cumdev = np.cumsum(sub - np.mean(sub))
            rs_vals.append((np.max(cumdev)-np.min(cumdev)) / (np.std(sub)+1e-10))
        ns = [hurst_win//4, hurst_win//2, hurst_win]
        if len(rs_vals) >= 2 and all(r > 0 for r in rs_vals):
            slope = np.polyfit(np.log(ns[:len(rs_vals)]), np.log(rs_vals), 1)[0]
            hurst_exp.iloc[i] = float(slope)
    f['hurst_exp'] = hurst_exp
    # FFT
    fft_win = bpd * 2
    fft_strength = pd.Series(np.nan, index=df.index, dtype=np.float32)
    for i in range(fft_win, len(close_arr)):
        seg_ret = np.diff(np.log(close_arr[i-fft_win:i] + 1e-10))
        fft_mag = np.abs(np.fft.rfft(seg_ret))
        fft_strength.iloc[i] = float(np.max(fft_mag[1:]) / (np.mean(fft_mag[1:])+1e-10))
    f['fft_strength']    = fft_strength.astype(np.float32)
    f['frac_diff_close'] = frac_diff_ffd(np.log(c.clip(lower=1e-10)), d=0.4)
    return f.replace([np.inf, -np.inf], np.nan).shift(1).astype(np.float32)

print(f'[Cell 3] Feature engineering ready. {len(FEATURE_COLS)} features defined.')

In [ ]:
# ── Cell 4: Auto-loop all batches — build + upload ALL symbols ───────────────
# Uploads each .parquet to Google Drive immediately after building,
# then deletes the local copy so disk never fills up.
# Resume-safe: files already in Drive are skipped.
import psutil

CHUNK_GC_EVERY = 25

total_batches  = -(-len(all_files) // BATCH_SIZE)
grand_built = grand_uploaded = grand_skipped = 0
t_total = time.time()

for BATCH_INDEX in range(total_batches):

    batch_files = all_files[BATCH_INDEX * BATCH_SIZE : (BATCH_INDEX + 1) * BATCH_SIZE]

    print(f'{"="*60}')
    print(f'  BATCH {BATCH_INDEX+1}/{total_batches}  '
          f'(symbols {BATCH_INDEX*BATCH_SIZE}–{BATCH_INDEX*BATCH_SIZE+len(batch_files)-1})')
    print(f'{"="*60}')

    already_uploaded = gdrive_list_files()
    print(f'  Already in Drive : {len(already_uploaded)} files')
    print(f'  Disk free        : {shutil.disk_usage("/kaggle/working").free / 1e9:.1f} GB\n')

    n_built = n_uploaded = n_skipped = 0
    t0 = time.time()

    for idx, fp in enumerate(batch_files, 1):
        out_name = fp.stem + '.parquet'
        symbol   = fp.stem

        # Already uploaded — skip
        if out_name in already_uploaded:
            n_skipped += 1
            continue

        out_path = Path(FEATURE_STORE_DIR).resolve() / out_name

        try:
            df = pd.read_parquet(fp)

            # Fix timestamp
            if not isinstance(df.index, pd.DatetimeIndex):
                ts_col = next((c for c in df.columns if c in ('time','timestamp','open_time')), None)
                if ts_col:
                    unit = 'ms' if pd.api.types.is_integer_dtype(df[ts_col]) else None
                    df.index = pd.to_datetime(df[ts_col], unit=unit, utc=True)
                    df = df.drop(columns=[ts_col])
                elif pd.api.types.is_integer_dtype(df.index):
                    df.index = pd.to_datetime(df.index, unit='ms', utc=True)
                else:
                    df.index = pd.to_datetime(df.index, utc=True)
            elif df.index.tz is None:
                df.index = df.index.tz_localize('UTC')

            if df.index.max().year < 2018:
                raise ValueError(f'Bad timestamp: max={df.index.max()}')
            df = df.sort_index()

            # Build features + forward return label
            feat = build_features(df)
            feat['future_ret'] = np.log(df['close'].shift(-HORIZON_BARS) / df['close'])
            feat['symbol']     = fp.stem
            feat[FEATURE_COLS + ['future_ret']] = \
                feat[FEATURE_COLS + ['future_ret']].astype(np.float32)
            feat.dropna(subset=['future_ret'] + FEATURE_COLS[:5], how='all') \
                .to_parquet(out_path, compression='zstd')
            n_built += 1

            # Upload to Google Drive
            gdrive_upload_file(out_path)
            already_uploaded.add(out_name)
            n_uploaded += 1

            # Delete local copy immediately to keep disk free
            out_path.unlink(missing_ok=True)

            del df, feat
            gc.collect()

        except Exception as e:
            print(f'    [ERR] {symbol}: {e}')
            continue

        if idx % CHUNK_GC_EVERY == 0 or idx == len(batch_files):
            elapsed = time.time() - t0
            ram_gb  = psutil.virtual_memory().used / 1e9
            disk_gb = shutil.disk_usage('/kaggle/working').used / 1e9
            print(f'    [{idx:2d}/{len(batch_files)}] built={n_built} up={n_uploaded} '
                  f'skip={n_skipped} | '
                  f'RAM={ram_gb:.1f}GB disk={disk_gb:.2f}GB elapsed={elapsed/60:.1f}min',
                  flush=True)
            gc.collect()

    grand_built    += n_built
    grand_uploaded += n_uploaded
    grand_skipped  += n_skipped
    print(f'  Batch {BATCH_INDEX+1} done — {n_built} built, {n_uploaded} uploaded, '
          f'{n_skipped} skipped\n')

# ════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ════════════════════════════════════════════════════════════════════════════
final_count = len(gdrive_list_files())
print('━'*60)
print(f'  ALL {total_batches} BATCHES COMPLETE')
print(f'  Built    : {grand_built}')
print(f'  Uploaded : {grand_uploaded}')
print(f'  Skipped  : {grand_skipped}   (already existed in Drive)')
print(f'  Total time : {(time.time()-t_total)/60:.1f} min')
print(f'  Files in Google Drive : {final_count}')
print('━'*60)
print()
print(f'  Drive folder: https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}')
print('  → Next: open azalyst_2_train.ipynb and set GDRIVE_FOLDER_ID to the same value.')
